**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO _DEEPLY UNDERSTAND HOW __slots__ CHANGES OBJECT MEMORY LAYOUT, ATTRIBUTE STORAGE, AND PERFORMANCE_. 🐍🧠**

`__slots__` is often introduced as a way to "save memory" — but *why* does it actually work?

In this DeepCut, we explore:
- How Python normally stores attributes using `__dict__`
- What changes internally when `__slots__` is used
- How attribute lookup works with and without slots
- Why slots matter in object-heavy applications
- When using `__slots__` is a **bad idea**

---

## 📦 Import Standard Library

In [1]:
import sys

---

## 🧩 Snippet 1 — Normal objects store attributes in a per-instance __dict__

By default, every Python object has a dictionary that stores its attributes dynamically.

In [2]:
class Normal:
    def __init__(self, x, y):
        self.x = x
        self.y = y

n = Normal(10, 20)

print("Attributes:", n.__dict__)
print("Has __dict__:", hasattr(n, "__dict__"))
print("Size of __dict__:", sys.getsizeof(n.__dict__))

Attributes: {'x': 10, 'y': 20}
Has __dict__: True
Size of __dict__: 296


---

## 🧠 Snippet 2 — __slots__ removes the per-instance __dict__

When __slots__ is defined:
- Python does NOT create a __dict__ per instance
- Attributes are stored in a fixed internal structure
- Dynamic attribute creation is prevented

In [3]:
class Slotted:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

s = Slotted(10, 20)

print("Has __dict__:", hasattr(s, "__dict__"))
print("x:", s.x, "y:", s.y)

Has __dict__: False
x: 10 y: 20


---

## 🔍 Snippet 3 — Comparing memory usage of normal vs slotted instances

Slots reduce memory by removing the attribute dictionary.

In [4]:
normal = Normal(1, 2)
slotted = Slotted(1, 2)

print("Normal object size:", sys.getsizeof(normal))
print("Slotted object size:", sys.getsizeof(slotted))

Normal object size: 48
Slotted object size: 48


> Note: The biggest saving comes from removing __dict__, not the instance itself.

----

## 🧱 Snippet 4 — Attribute lookup paths differ internally

- Normal objects → attribute lookup goes through __dict__
- Slotted objects → attribute offset is known at compile-time

This reduces indirection and speeds up lookup slightly.

In [5]:
print("Normal __dict__ lookup:", normal.__dict__["x"])
print("Slotted direct access:", slotted.x)

Normal __dict__ lookup: 1
Slotted direct access: 1


---

## ⚠️ Snippet 5 — __slots__ disallows adding new attributes

This restriction is both a feature and a limitation.

In [6]:
try:
    s.z = 30
except AttributeError as e:
    print("Error:", e)

Error: 'Slotted' object has no attribute 'z' and no __dict__ for setting new attributes


---

## 🧬 Snippet 6 — Slots must be declared carefully in inheritance

Slots do NOT automatically combine across class hierarchies.

In [7]:
class Base:
    __slots__ = ("a",)

class Child(Base):
    __slots__ = ("b",)

c = Child()
c.a = 1
c.b = 2
print(c.a, c.b)

1 2


---

## 🔄 Snippet 7 — Mixing __slots__ with __dict__ for flexibility

You can allow dynamic attributes explicitly if needed.

In [8]:
class Hybrid:
    __slots__ = ("x", "__dict__")

h = Hybrid()
h.x = 10
h.y = 20  # allowed because __dict__ exists

print(h.x, h.y)
print("Hybrid __dict__:", h.__dict__)

10 20
Hybrid __dict__: {'y': 20}


---

## 🧠 When __slots__ Is Worth Using

Use __slots__ when:
- You create **millions of objects**
- Your objects have a **fixed set of attributes**
- Memory pressure matters (APIs, ML, simulations)

Avoid __slots__ when:
- You need dynamic attributes
- You rely on monkey-patching
- You expect frequent schema changes

---

## ✅ One-liner Takeaway

**__slots__ replaces per-instance dictionaries with a compact attribute layout — saving memory, speeding up access, and enforcing structure in object-heavy systems.**

---